## Model #1 Prep - 4x ETFS + Headline and Article Content Sentiment Scores 

In [2]:
import pandas as pd
import json

In [13]:
NEWS_DATA_FOLDER = './data/articles_sentiment_processed/'    # folder with processed sentiment news
qqq = pd.read_csv("./data/raw/QQQ.csv")
spx = pd.read_csv("./data/raw/GSPC.csv")
spy = pd.read_csv("./data/raw/SPY.csv")
es = pd.read_csv("./data/raw/ES=F.csv")

In [ ]:
for df in [qqq, spx, spy, es]:
    df["Date"] = pd.to_datetime(df["Date"])

qqq = qqq.rename(columns=lambda x: f"qqq_{x}" if x != "Date" else x)
spx = spx.rename(columns=lambda x: f"spx_{x}" if x != "Date" else x)
spy = spy.rename(columns=lambda x: f"spy_{x}" if x != "Date" else x)
es = es.rename(columns=lambda x: f"es_{x}" if x != "Date" else x)

stock_data = spy.copy()  # Start with SPY
for df in [qqq, spx, es]:
    data = pd.merge(data, df, on="Date", how="inner")  # only keep dates common to all ETFs

# Sort by date just in case
stock_data = data.sort_values("Date")

# Quick look
print(stock_data.head())

        Date  spy_Close   spy_High    spy_Low   spy_Open  spy_Volume  \
0 2010-01-04  85.768440  85.813847  84.391060  85.041910   118944600   
1 2010-01-05  85.995476  86.033318  85.405171  85.715463   111579900   
2 2010-01-06  86.056046  86.267949  85.844142  85.912251   116074400   
3 2010-01-07  86.419289  86.525241  85.654916  85.897093   131091100   
4 2010-01-08  86.706879  86.744721  86.018191  86.192253   126402800   

   qqq_Close   qqq_High    qqq_Low   qqq_Open  ...    spx_Close     spx_High  \
0  40.485809  40.546864  40.354987  40.407318  ...  1132.989990  1133.869995   
1  40.485809  40.555584  40.259048  40.459645  ...  1136.520020  1136.630005   
2  40.241611  40.599198  40.180560  40.468376  ...  1137.140015  1139.189941   
3  40.267796  40.355014  40.049755  40.302683  ...  1141.689941  1142.459961   
4  40.599197  40.599197  40.058457  40.180559  ...  1144.979980  1145.390015   

       spx_Low     spx_Open  spx_Volume  es_Close  es_High   es_Low  es_Open  \
0  111

In [ ]:
# Create a "future close" shifted -1 day
stock_data["spy_Close_future"] = data["spy_Close"].shift(-1)

# Create a binary "direction" label: 1 if up, 0 if down
stock_data["target"] = (data["spy_Close_future"] > data["spy_Close"]).astype(int)

# Drop the last row because it will have NaN in 'spy_close_future'
stock_data = data.dropna()
stock_data = stock_data.drop(columns=["spy_Close_future"])


# Quick look at target distribution
print(stock_data["target"].value_counts(normalize=True))


target
1    0.55296
0    0.44704
Name: proportion, dtype: float64


In [ ]:
import os
import pandas as pd

# Path to your news data directory
news_parts_dir = "./data/processed/articles/"

# Load and concatenate all CSV files
news_df = pd.concat(
    [pd.read_csv(os.path.join(news_parts_dir, f)) for f in sorted(os.listdir(news_parts_dir)) if f.endswith(".csv")],
    ignore_index=True
)

# Convert 'date_published' to datetime, and drop rows with missing essential data
news_df["date_published"] = pd.to_datetime(news_df["date_published"], errors="coerce")
news_df = news_df.dropna(subset=["article_content", "date_published"])

# 🧠 Now average separately:
daily_headline_sentiment = news_df.groupby(news_df["date_published"].dt.date)["headline_sentiment"].mean().reset_index()
daily_article_sentiment = news_df.groupby(news_df["date_published"].dt.date)["article_sentiment"].mean().reset_index()

# Rename for clarity
daily_headline_sentiment.columns = ["date", "avg_headline_sentiment"]
daily_article_sentiment.columns = ["date", "avg_article_sentiment"]

# Merge them together
daily_sentiment = pd.merge(daily_headline_sentiment, daily_article_sentiment, on="date")

# Convert date to datetime
daily_sentiment["date"] = pd.to_datetime(daily_sentiment["date"])

# Preview
print(daily_sentiment.head())

,Date,spy_Close,spy_High,spy_Low,spy_Open,spy_Volume,qqq_Close,qqq_High,qqq_Low,qqq_Open,...,spx_High,spx_Low,spx_Open,spx_Volume,es_Close,es_High,es_Low,es_Open,es_Volume,target
0,2010-01-04,85.768440,85.813847,84.391060,85.041910,118944600,40.485809,40.546864,40.354987,40.407318,...,1133.869995,1116.560059,1116.560059,3991400000,1128.750000,1129.75,1113.25,1113.75,1291254,1
1,2010-01-05,85.995476,86.033318,85.405171,85.715463,111579900,40.485809,40.555584,40.259048,40.459645,...,1136.630005,1129.660034,1132.660034,2491020000,1132.250000,1133.00,1125.00,1128.50,1378593,1
2,2010-01-06,86.056046,86.267949,85.844142,85.912251,116074400,40.241611,40.599198,40.180560,40.468376,...,1139.189941,1133.949951,1135.709961,4972660000,1133.000000,1135.50,1127.25,1132.00,1259921,1
3,2010-01-07,86.419289,86.525241,85.654916,85.897093,131091100,40.267796,40.355014,40.049755,40.302683,...,1142.459961,1131.319946,1136.270020,5270680000,1137.500000,1138.75,1127.00,1133.00,1567025,1
4,2010-01-08,86.706879,86.744721,86.018191,86.192253,126402800,40.599197,40.599197,40.058457,40.180559,...,1145.390015,1136.219971,1140.520020,4389590000,1141.500000,1141.75,1131.00,1137.25,1527666,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3762,2024-12-17,600.456665,601.331088,599.065582,600.357324,55773500,534.140808,535.815598,532.475997,534.699072,...,6057.680176,6035.189941,6052.549805,4544500000,6053.750000,6079.25,6040.75,6077.00,1113325,0
3763,2024-12-18,582.560974,602.563224,582.173434,600.148646,108248700,514.870667,535.217499,513.415226,533.492875,...,6070.669922,5867.790039,6047.649902,5246960000,5872.250000,6074.50,5840.00,6053.50,847452,0
3764,2024-12-19,582.382080,589.238335,582.133666,587.608723,85919500,512.577820,520.144344,512.238907,519.576101,...,5935.520020,5866.069824,5912.709961,4896880000,5868.750000,5938.00,5865.50,5881.75,532081,1
3765,2024-12-20,589.377075,593.963255,579.167735,580.025202,125716700,517.053955,523.194915,507.713004,508.859437,...,5982.060059,5832.299805,5842.000000,8223220000,5840.259766,5879.50,5800.75,5879.50,2340873,1


In [ ]:
merged_df = pd.merge(stock_data, daily_sentiment, on="date", how="left")
merged_df["avg_sentiment"] = merged_df["avg_sentiment"].fillna(0)
merged_df = merged_df.sort_values("date")
merged_df.head()